# Data Challenge — Face Occlusion (Colab training)

Phase 2 of the hybrid pipeline. Runs the real training on Colab GPU.

**Before running**: pick a GPU runtime (Runtime → Change runtime type → T4 or better).

## 1. Setup: clone the repo and install deps

In [ ]:
# Option A: clone from GitHub (replace with your repo URL)
# !git clone https://github.com/<your-username>/data-challenge-42.git
# %cd data-challenge-42

# Option B: upload a zip of the src/ + scripts/ folders via the Colab file pane,
# then unzip:
# !unzip -q code.zip

In [ ]:
!pip -q install pandas pillow torch torchvision tqdm matplotlib
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Get the data

Pick ONE of the options below depending on where you stored the images.

In [ ]:
# Option A: mount Google Drive (images uploaded there as crops/ and occlusion_datasets/)
from google.colab import drive
drive.mount('/content/drive')

# Edit the paths below to point at your Drive folders
DATA_DIR = '/content/drive/MyDrive/data-challenge-42/occlusion_datasets'
IMAGE_DIR = '/content/drive/MyDrive/data-challenge-42/crops'

import os
print('train.csv exists:', os.path.exists(f'{DATA_DIR}/train.csv'))
print('crops/ exists:', os.path.isdir(IMAGE_DIR))

In [ ]:
# Option B: upload zip directly to Colab local disk (faster I/O than Drive)
# !cp /content/drive/MyDrive/data-challenge-42/crops.zip /content/
# !unzip -q /content/crops.zip -d /content/
# DATA_DIR = '/content/occlusion_datasets'
# IMAGE_DIR = '/content/crops'

## 3. Train

Recommended starting point: `resnet50`, 8 epochs, batch 128, lr 3e-4. On a T4 expect ~10-15 min/epoch.

In [ ]:
!python scripts/train.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --backbone resnet50 \
  --epochs 8 \
  --batch-size 128 \
  --lr 3e-4 \
  --num-workers 2

## 4. Generate the submission file

In [ ]:
!python scripts/infer.py \
  --data-dir $DATA_DIR \
  --image-dir $IMAGE_DIR \
  --checkpoint checkpoints/resnet50_best.pt \
  --backbone resnet50 \
  --output test_predictions.csv

In [ ]:
# Download the submission to your local machine
from google.colab import files
files.download('test_predictions.csv')

## 5. Ideas for the next iteration

- Bigger backbone: `efficientnet_b0` → `efficientnet_b3`, or a ViT (`timm` library).
- Test-time augmentation: average prediction on image and its horizontal flip.
- Ensemble: train 2-3 different backbones, average their predictions.
- Synthetic augmentation for the high-occlusion tail (only 36 samples > 0.5 in train) — overlay rectangles/masks with known area.
- Tune `WeightedMSELoss(balance_gender=...)` and the balanced sampler — try with one only, both, or neither.